In [57]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

## Data Processing

In [58]:
data = pd.read_csv('course_lead_scoring.csv')
data.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [59]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               1334 non-null   object 
 1   industry                  1328 non-null   object 
 2   number_of_courses_viewed  1462 non-null   int64  
 3   annual_income             1281 non-null   float64
 4   employment_status         1362 non-null   object 
 5   location                  1399 non-null   object 
 6   interaction_count         1462 non-null   int64  
 7   lead_score                1462 non-null   float64
 8   converted                 1462 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 102.9+ KB


In [60]:
data[data.select_dtypes('object').columns] = data.select_dtypes('object').fillna('NA')
data[data.select_dtypes(['integer', 'float']).columns] = data.select_dtypes(['integer', 'float']).fillna(0)

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               1462 non-null   object 
 1   industry                  1462 non-null   object 
 2   number_of_courses_viewed  1462 non-null   int64  
 3   annual_income             1462 non-null   float64
 4   employment_status         1462 non-null   object 
 5   location                  1462 non-null   object 
 6   interaction_count         1462 non-null   int64  
 7   lead_score                1462 non-null   float64
 8   converted                 1462 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 102.9+ KB


## Question 1
What is the most frequent observation (mode) for the column industry?

NA
technology
healthcare
retail

In [61]:
data['industry'].value_counts()

industry
retail           203
finance          200
other            198
healthcare       187
education        187
technology       179
manufacturing    174
NA               134
Name: count, dtype: int64

In [62]:
data[data.select_dtypes(['integer', 'float']).columns].corr().round(2)

,number_of_courses_viewed,annual_income,interaction_count,lead_score,converted
number_of_courses_viewed,1.00,0.01,-0.02,-0.00,0.44
annual_income,0.01,1.00,0.03,0.02,0.05
interaction_count,-0.02,0.03,1.00,0.01,0.37
lead_score,-0.00,0.02,0.01,1.00,0.19
converted,0.44,0.05,0.37,0.19,1.00


In [67]:
# Split your data in train/val/test sets with 60%/20%/20% distribution.
X = data.drop(columns=['converted'])
y = data['converted']
full_X_train, X_raw_test = train_test_split(X, test_size=0.2, random_state=42)
full_y_train, y_test = train_test_split(y, test_size=0.2, random_state=42)
X_raw_train, X_raw_val = train_test_split(full_X_train, test_size=0.33, random_state=42)
y_train, y_val = train_test_split(full_y_train, test_size=0.33, random_state=42)

In [68]:
data.shape, X_raw_train.shape, X_raw_val.shape, X_raw_test.shape

((1462, 9), (783, 8), (386, 8), (293, 8))

In [69]:
from sklearn.metrics import mutual_info_score
def mutual_info(series):
    return mutual_info_score(series, y_train).round(2)

In [70]:
for c in X_raw_train.select_dtypes('object').columns:
    if c == 'converted':
        continue
    print(c, mutual_info(X_raw_train[c]))

lead_source 0.03
industry 0.01
employment_status 0.01
location 0.0


## One-hot encoding using DictVectorizer

In [71]:
from sklearn.feature_extraction import DictVectorizer

In [72]:
train_dict = X_raw_train.to_dict(orient='records')
val_dict = X_raw_val.to_dict(orient='records')
test_dict = X_raw_test.to_dict(orient='records')

In [73]:
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)
X_test = dv.transform(test_dict)

In [74]:
X_train.shape

(783, 31)

In [75]:
len(dv.feature_names_)

31

## Fitting the Logit Model

In [76]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [77]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [78]:
pred_val = model.predict(X_val)
pred_val

array([1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1,
       0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0,
       1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1,
       0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1,
       1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1,

In [79]:
from sklearn.metrics import accuracy_score
round(accuracy_score(y_val, pred_val), 2)

0.71

## Question 5
Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference?

In [80]:
# Main model with all features and no rounding
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
pred_val = model.predict(X_val)
main_accuracy = accuracy_score(y_val, pred_val)

In [81]:
feature_accuracy_diff = {}
for i, c in enumerate(X_raw_train.columns):

    train_dict = X_raw_train.drop(columns=c).to_dict(orient='records')
    val_dict = X_raw_val.drop(columns=c).to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dict)
    X_val = dv.transform(val_dict)

    # fit the model without this feature
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_val).round(2)
    feature_accuracy_diff[c] = main_accuracy - accuracy_score(y_val, pred)

In [83]:
pd.DataFrame(feature_accuracy_diff, index=[0]).T.rename(columns={0: 'Accuracy Drop'}).sort_values('Accuracy Drop', ascending=True)

,Accuracy Drop
annual_income,-0.147668
industry,0.000000
location,0.000000
lead_score,0.002591
lead_source,0.007772
employment_status,0.010363
number_of_courses_viewed,0.142487
interaction_count,0.145078


## Question 6
Now let's train a regularized logistic regression.
Let's try the following values of the parameter C: [0.01, 0.1, 1, 10, 100].
Train models using all the features as in Q4.
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these C leads to the best accuracy on the validation set?

In [84]:
C_values = [0.01, 0.1, 1, 10, 100]
results = {}

for C in C_values:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    acc = accuracy_score(y_val, pred)
    results[C] = round(acc, 3)
    print(f"C={C}: accuracy={results[C]}")

# Find best C
best_C = min([c for c in C_values if results[c] == max(results.values())])
print(f"\nBest C: {best_C} with accuracy: {results[best_C]}")

C=0.01: accuracy=0.71
C=0.1: accuracy=0.71
C=1: accuracy=0.71
C=10: accuracy=0.71
C=100: accuracy=0.71

Best C: 0.01 with accuracy: 0.71
